# Retail Inventory & Sales Analytics
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** Synthetic 2025 retail transactions across 6 stores.

This notebook validates the data, creates business metrics, explores sales trends, and flags stock risks. Run from this notebook's directory or update `DATA_DIR` below.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../dataset")  # change if your notebook is elsewhere
sales = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "products.csv")
inventory = pd.read_csv(DATA_DIR / "inventory.csv", parse_dates=["snapshot_date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")

print("Sales:", sales.shape, "Products:", products.shape,
      "Inventory:", inventory.shape, "Stores:", stores.shape)
sales.head()


## 1. Data quality checks

In [ ]:
# Missing values, duplicate keys, and basic validity checks
for name, df in {"sales": sales, "products": products,
                 "inventory": inventory, "stores": stores}.items():
    print(f"\\n{name.upper()} missing values\\n", df.isna().sum())

print("Duplicate product IDs:", products.product_id.duplicated().sum())
print("Duplicate store IDs:", stores.store_id.duplicated().sum())
print("Invalid quantities:", (sales.quantity_sold <= 0).sum())
print("Negative sales:", (sales.sales_amount < 0).sum())

# Validate line amount calculation within rounding tolerance
calc = sales.quantity_sold * sales.unit_price * (1-sales.discount_pct)
print("Rows with sales amount mismatch:", ((calc-sales.sales_amount).abs() > 0.02).sum())


## 2. Build analysis dataset and KPIs

In [ ]:
df = (sales.merge(products, on="product_id", suffixes=("", "_product"))
           .merge(stores, on="store_id"))
df["gross_profit"] = df["sales_amount"] - df["cost_amount"]
df["gross_margin_pct"] = np.where(df.sales_amount != 0,
                                  df.gross_profit / df.sales_amount * 100, 0)
df["month"] = df.order_date.dt.to_period("M").astype(str)

kpis = {
    "Revenue": df.sales_amount.sum(),
    "Gross profit": df.gross_profit.sum(),
    "Gross margin %": df.gross_profit.sum()/df.sales_amount.sum()*100,
    "Units sold": df.quantity_sold.sum(),
    "Orders": df.order_id.nunique(),
    "Average order value": df.sales_amount.sum()/df.order_id.nunique()
}
pd.Series(kpis).round(2)


## 3. Sales trends

In [ ]:
monthly = df.groupby("month", as_index=False).agg(
    revenue=("sales_amount","sum"),
    gross_profit=("gross_profit","sum"),
    units=("quantity_sold","sum")
)
plt.figure(figsize=(12,5))
sns.lineplot(data=monthly, x="month", y="revenue", marker="o")
plt.title("Monthly Revenue Trend"); plt.xlabel("Month"); plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

category = df.groupby("category", as_index=False).agg(
    revenue=("sales_amount","sum"), units=("quantity_sold","sum"),
    profit=("gross_profit","sum")
).sort_values("revenue", ascending=False)
plt.figure(figsize=(10,5))
sns.barplot(data=category, x="revenue", y="category")
plt.title("Revenue by Category"); plt.xlabel("Revenue (INR)"); plt.ylabel("")
plt.tight_layout(); plt.show()
category


## 4. Product performance

In [ ]:
product_perf = df.groupby(
    ["product_id","product_name","category"], as_index=False
).agg(units_sold=("quantity_sold","sum"),
      revenue=("sales_amount","sum"),
      gross_profit=("gross_profit","sum"))
top10 = product_perf.nlargest(10, "revenue")
bottom10 = product_perf.nsmallest(10, "units_sold")
display(top10)
display(bottom10)

plt.figure(figsize=(10,5))
sns.barplot(data=top10, x="revenue", y="product_name")
plt.title("Top 10 Products by Revenue"); plt.tight_layout(); plt.show()


## 5. Inventory risk analysis

In [ ]:
recent = sales[sales.order_date >= "2025-10-01"].groupby(
    ["product_id","store_id"], as_index=False
).agg(units_last_3m=("quantity_sold","sum"))
recent["avg_monthly_units"] = recent.units_last_3m / 3

inv_analysis = (inventory.merge(products, on="product_id")
                .merge(stores, on="store_id")
                .merge(recent[["product_id","store_id","avg_monthly_units"]],
                       on=["product_id","store_id"], how="left"))
inv_analysis["avg_monthly_units"] = inv_analysis.avg_monthly_units.fillna(0)
inv_analysis["months_of_cover"] = np.where(
    inv_analysis.avg_monthly_units > 0,
    inv_analysis.stock_quantity / inv_analysis.avg_monthly_units, np.nan)
inv_analysis["stock_status"] = np.select(
    [inv_analysis.stock_quantity <= inv_analysis.reorder_level,
     inv_analysis.months_of_cover > 8],
    ["Low stock","Overstock"], default="Healthy"
)
display(inv_analysis.stock_status.value_counts().rename("product-store combinations"))
display(inv_analysis[inv_analysis.stock_status=="Low stock"].head(20))


## 6. Inventory value and store performance

In [ ]:
inv_analysis["inventory_cost_value"] = (
    inv_analysis.stock_quantity * inv_analysis.unit_cost
)
print("Total inventory cost value (INR):",
      round(inv_analysis.inventory_cost_value.sum(), 2))

store_perf = df.groupby(["store_name","city"], as_index=False).agg(
    revenue=("sales_amount","sum"),
    orders=("order_id","nunique"),
    units=("quantity_sold","sum")
)
store_perf["average_order_value"] = store_perf.revenue/store_perf.orders
display(store_perf.sort_values("revenue", ascending=False))


## 7. Export analysis outputs

These files can be used as Power BI supporting tables or project evidence.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
monthly.to_csv(OUTPUT_DIR/"monthly_sales_summary.csv", index=False)
category.to_csv(OUTPUT_DIR/"category_performance.csv", index=False)
product_perf.to_csv(OUTPUT_DIR/"product_performance.csv", index=False)
inv_analysis.to_csv(OUTPUT_DIR/"inventory_risk_analysis.csv", index=False)
store_perf.to_csv(OUTPUT_DIR/"store_performance.csv", index=False)
print("Analysis outputs saved to", OUTPUT_DIR.resolve())
